# Workflow quickstart

Interactive setup, provider smoke test, and representative end-to-end Table2Text run. Configure models in `table2text_pydanticai/.env` before execution.


## Environment and imports


In [ ]:
from pathlib import Path
from dataclasses import replace
from IPython.display import Markdown, display

PROJECT = Path("/Users/realgobs/Documents/MScproject/table2text_pydanticai")
INPUTS = PROJECT / "inputs"

%cd {PROJECT}

from table2text import Settings, Table2TextWorkflow
from table2text.schemas import AuditMode

## Inspect local inputs


In [ ]:
# Check input files
for path in sorted(INPUTS.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT))

## Verify the configured provider


In [ ]:
import os, getpass
from pydantic_ai import Agent

if not os.getenv("DEEPSEEK_API_KEY"):
    os.environ["DEEPSEEK_API_KEY"] = getpass.getpass("DeepSeek API key: ")

agent = Agent("deepseek:deepseek-chat", output_type=str)
result = await agent.run("Reply with exactly: OK")
print(result.output)

## Run the workflow


In [ ]:
from pathlib import Path

from table2text import Settings, Table2TextWorkflow
from table2text.schemas import AuditMode, EvaluationFieldPolicy

project_dir = Path(
    "/Users/realgobs/Documents/MScproject/table2text_pydanticai"
)

settings = Settings(
    use_llm=True,
    structured_output_mode="prompted",
    max_total_tokens=160_000,
    max_agent_requests=8,

    data_understanding_model="deepseek:deepseek-chat",
    orchestrator_model="deepseek:deepseek-chat",
    evidence_model="deepseek:deepseek-chat",
    verifier_model="deepseek:deepseek-chat",
    writer_model="deepseek:deepseek-chat",
    auditor_model="deepseek:deepseek-chat",

    output_dir=project_dir / "runs_notebook",
)

workflow = Table2TextWorkflow(settings)

field_policy = EvaluationFieldPolicy(
    operational_input_paths=[
        "game_id",
        "date",
        "location",
        "overtime",
        "teams",
    ],
    held_out_reference_paths=["summary"],
    metadata_paths=["basketballreference"],
)

result = await workflow.run(
    inputs=[project_dir / "inputs/basketball_data.json"],
    request="Understand the dataset and give a neutral report.",
    audit_mode=AuditMode.INTERNAL,
    evaluation_field_policy=field_policy,
)

print("Run ID:", result.run_id)
print("Release status:", result.release_status.value)
print("Final audit:", result.final_audit.decision.value)
print("Writer mode:", result.raw_writer_output.writer_mode)
print("Verified insights:", len(result.insight_ledger.verified_insights))
print(
    "Insight fallback:",
    result.insight_ledger.fallback_reason or "None",
)
print(
    "Verifier notes:",
    result.fact_ledger.verifier_notes or ["None"],
)
print(
    "Report:",
    project_dir / "runs_notebook" / result.run_id / "final_report.md",
)

## Inspect support and release metadata


In [ ]:
print(
    "Raw writer mode:",
    result.raw_writer_output.writer_mode,
)
print(
    "Primary-evaluation eligible:",
    result.raw_writer_output
    .eligible_for_primary_evaluation,
)
print(
    "Support entries:",
    len(
        result.raw_writer_output
        .sentence_support
    ),
)
print(
    "Quality revision used:",
    result.quality_revised_writer_output
    is not None,
)

## Inspect the quality assessment


In [ ]:
quality = (
    result.final_audit
    .quality_assessment
)

print("Quality status:", quality.status.value)

print("\nQuality findings:")
for finding in quality.findings:
    print("-", finding)

print("\nQuality recommendations:")
for recommendation in quality.recommendations:
    print("-", recommendation)

print("\nMethodological warnings:")
for warning in (
    result.final_audit
    .methodological_warnings
):
    print("-", warning)

print("\nAnnotations:")
for annotation in (
    result.final_audit.annotations
):
    print(
        annotation.severity.value,
        annotation.subtype,
        "=>",
        annotation.explanation,
    )